In [ ]:
import sqlite3
import sys
from pathlib import Path
import pandas as pd

# Add src to system path to import our custom analyzer
sys.path.append(str(Path.cwd().parent.parent / "src"))
from sentiment.analyser import SentimentEngine

# 1. Connect to SQLite
db_path = Path.cwd().parent.parent / "data" / "processed" / "cryptocurrency-110826.db"
conn = sqlite3.connect(db_path)

# 2. Extract posts with combined text
posts_df = pd.read_sql("""
    SELECT 
        id AS post_id,
        timestamp,
        title,
        description,
        (title || ' ' || COALESCE(description, '')) AS full_text,
        upvotes,
        comments_count
    FROM posts
    ORDER BY timestamp DESC
    LIMIT 20;
""", conn)

# 3. Extract comments for these posts
post_ids = tuple(posts_df['post_id'].tolist())
comments_df = pd.read_sql(f"""
    SELECT post_id as post_id, id AS comment_id, author, score as score, text as text
    FROM comments
    WHERE post_id IN {post_ids} AND text NOT IN ('[deleted]', '[removed]')""", conn)

# print(comments_df.iloc[0:10].TEXT)

# 4. Initialize Sentiment Engine
engine = SentimentEngine(load_finbert=True)

# 5. Score Posts with VADER and FinBERT
posts_df['vader_compound'] = posts_df['full_text'].apply(lambda t: engine.score_vader(t)['compound'])
finbert_post_scores = engine.score_finbert_batch(posts_df['full_text'].tolist())
posts_df['finbert_net'] = [s['finbert_net'] for s in finbert_post_scores]

#print(posts_df)
#print(comments_df)
#print(comments_df.columns.tolist())
#print(comments_df.groupby('post_id').head())
#print(f"\n posts_df.columns: {posts_df.columns.tolist()}")



# 6. Score Comments & Aggregate
if not comments_df.empty:
    comments_df['vader_compound'] = comments_df['text'].apply(lambda t: engine.score_vader(t)['compound'])
    
    # Calculate weighted comment mood (weighted by comment upvotes)
    def calculate_weighted_sentiment(group):
        # Clip negative scores to 0 to prevent inverted weights
        weights = group['score'].clip(lower=0) + 1  
        return (group['vader_compound'] * weights).sum() / weights.sum()

    comment_summary = comments_df.groupby('post_id').apply(calculate_weighted_sentiment).reset_index(name='comments_avg_sentiment')
    
    # Merge back to posts
    posts_df = posts_df.merge(comment_summary, on='post_id', how='left')
    posts_df['consensus_divergence'] = posts_df['vader_compound'] - posts_df['comments_avg_sentiment'].fillna(0)

# Display results
posts_df[['TITLE', 'UPVOTES', 'vader_compound', 'finbert_net', 'consensus_divergence']].head(10)

